# Experiment 1: RAG-based Anime Recommendation System

## 1. Objectives & Context Preservation
In this experiment, we build a Retrieval-Augmented Generation (RAG) system with tool-calling capabilities for anime recommendations.

### Key Problem: How Data is Split so Context is Never Broken
- In standard RAG tutorials, texts are split using `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`.
- **Why character splitting fails for anime data**:
  - An anime entry contains a **Title**, an **Overview/Synopsis**, **Genres**, and a **Score**.
  - If split by character count, the synopsis gets cut across multiple chunks. Subsequent chunks lose the anime's **Title** and **Genres**.
  - The retriever may fetch a chunk like *"...and saved the village from destruction."*, leaving the LLM unable to identify which anime it refers to.
- **The Solution (Atomic 1 Anime = 1 Document)**:
  - Each anime entry in `anime_with_synopsis.csv` is maintained as **one atomic LangChain `Document` chunk**.
  - Across our 269 anime titles, the average length is ~132 words (778 characters) and the maximum length is 261 words (1532 characters, ~340 tokens).
  - All anime documents comfortably fit within the context window of standard embedding models (e.g., `all-MiniLM-L6-v2` has a context length of 256–512 tokens).
  - Therefore, **no character splitting is performed**, guaranteeing 100% intact context.

In [1]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# Load environment variables (searches current and parent directories for .env)
load_dotenv(find_dotenv(), override=True)
groq_key = os.getenv("GROQ_API_KEY", "")
os.environ["GROQ_API_KEY"] = groq_key

print("Environment loaded successfully.")
print(f"Groq API Key configured: {'Yes' if groq_key else 'No'}")

c:\Users\DELL\Desktop\anime recommandation system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment loaded successfully.
Groq API Key configured: Yes


In [2]:
# Initialize LLM via Groq provider
# Using openai/gpt-oss-20b (high performance & reliable within Groq free-tier OTPM limits)
model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.2,
    max_tokens=600
)

# Verify model connectivity with a short test invocation
response = model.invoke("Reply in one short sentence: Ready to recommend anime!")
print("LLM Response:", response.content)

LLM Response: Sure, what genre or mood are you in?


## 2. Data Ingestion & Whole-Anime Document Splitting

Here we define `AnimeDataLoader`. It reads the raw anime CSV, formats each record into a unified semantic block (`Title`, `Overview`, `Genres`), attaches rich metadata (`title`, `genres`, `score`, `mal_id`), and returns 1 complete `Document` per anime.

In [3]:
from typing import List, Optional

class AnimeDataLoader:
    def __init__(self, original_csv: str, processed_csv: Optional[str] = None):
        self.original_csv = Path(original_csv)
        self.processed_csv = Path(processed_csv) if processed_csv else None

    def load_and_split(self) -> List[Document]:
        """
        Loads raw anime CSV and returns 1 Document per anime to keep context 100% intact.
        """
        if not self.original_csv.exists():
            raise FileNotFoundError(f"Source CSV not found: {self.original_csv}")

        df = pd.read_csv(self.original_csv, encoding="utf-8", on_bad_lines="skip").dropna()

        # Handle column spelling variations (sypnopsis vs synopsis)
        synopsis_col = "sypnopsis" if "sypnopsis" in df.columns else "synopsis"
        required_cols = {"Name", "Genres", synopsis_col}
        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns in CSV: {missing}")

        # Combine anime fields into a structured semantic block
        df["combined_info"] = (
            "Title: " + df["Name"].astype(str) + "\n" +
            "Overview: " + df[synopsis_col].astype(str) + "\n" +
            "Genres: " + df["Genres"].astype(str)
        )

        # Optionally save processed CSV
        if self.processed_csv:
            self.processed_csv.parent.mkdir(parents=True, exist_ok=True)
            df[["combined_info"]].to_csv(self.processed_csv, index=False, encoding="utf-8")

        # Atomic document splitting: 1 row = 1 Document chunk
        documents = [
            Document(
                page_content=row["combined_info"],
                metadata={
                    "title": str(row["Name"]),
                    "genres": str(row["Genres"]),
                    "score": str(row.get("Score", "N/A")),
                    "mal_id": int(row["MAL_ID"]) if "MAL_ID" in df.columns and pd.notna(row["MAL_ID"]) else None,
                }
            )
            for _, row in df.iterrows()
        ]

        return documents

In [4]:
# Resolve CSV paths dynamically (works from notebook dir or project root)
candidate_paths = [
    Path("anime_with_synopsis.csv"),
    Path("expreiments/anime_with_synopsis.csv"),
    Path("../expreiments/anime_with_synopsis.csv")
]
csv_path = next((p for p in candidate_paths if p.exists()), None)
if not csv_path:
    raise FileNotFoundError("Could not find anime_with_synopsis.csv")

processed_csv_path = csv_path.parent / "anime_updated.csv"

# Load documents
loader = AnimeDataLoader(
    original_csv=str(csv_path),
    processed_csv=str(processed_csv_path)
)
anime_docs = loader.load_and_split()

print(f"Total anime documents created: {len(anime_docs)}")

# Verify chunk sizes & context preservation
lengths = [len(doc.page_content) for doc in anime_docs]
print(f"Document character lengths: Min={min(lengths)}, Mean={sum(lengths)/len(lengths):.1f}, Max={max(lengths)}")

print("\n=== Sample Complete Document ===")
print(anime_docs[0].page_content)
print("\n=== Sample Document Metadata ===")
print(anime_docs[0].metadata)

Total anime documents created: 269
Document character lengths: Min=319, Mean=867.8, Max=1630

=== Sample Complete Document ===
Title: Cowboy Bebop
Overview: In the year 2071, humanity has colonized several of the planets and moons of the solar system leaving the now uninhabitable surface of planet Earth behind. The Inter Solar System Police attempts to keep peace in the galaxy, aided in part by outlaw bounty hunters, referred to as "Cowboys." The ragtag team aboard the spaceship Bebop are two such individuals. Mellow and carefree Spike Spiegel is balanced by his boisterous, pragmatic partner Jet Black as the pair makes a living chasing bounties and collecting rewards. Thrown off course by the addition of new members that they meet in their travels—Ein, a genetically engineered, highly intelligent Welsh Corgi; femme fatale Faye Valentine, an enigmatic trickster with memory loss; and the strange computer whiz kid Edward Wong—the crew embarks on thrilling adventures that unravel each memb

## 3. Vector Database & Embeddings with Chroma

We use `sentence-transformers/all-MiniLM-L6-v2` for dense embeddings and `Chroma` as our local vector database.
We provide `VectorStoreBuilder` to build a clean index (avoiding stale fragments) or load an existing store.

In [5]:
import shutil

class VectorStoreBuilder:
    def __init__(self, persist_dir: str = "chroma_db", embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.persist_dir = Path(persist_dir).resolve()
        self.embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

    def build_and_save_vector_store(self, documents: List[Document], overwrite: bool = True) -> Chroma:
        """
        Builds Chroma vector store. If overwrite is True, cleanly recreates the store
        to ensure no stale or fragmented historical chunks persist.
        """
        if overwrite and self.persist_dir.exists():
            shutil.rmtree(self.persist_dir)
        self.persist_dir.mkdir(parents=True, exist_ok=True)

        vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            persist_directory=str(self.persist_dir)
        )
        return vectorstore

    def load_vector_store(self) -> Chroma:
        """Loads existing Chroma vector store from disk."""
        if not self.persist_dir.exists():
            raise FileNotFoundError(f"Chroma store at {self.persist_dir} does not exist.")
        return Chroma(
            persist_directory=str(self.persist_dir),
            embedding_function=self.embeddings
        )

In [6]:
persist_dir = csv_path.parent / "chroma_db"
vector_builder = VectorStoreBuilder(persist_dir=str(persist_dir))

# Build clean vector store with our 269 intact anime documents
vectorstore = vector_builder.build_and_save_vector_store(anime_docs, overwrite=True)
print(f"Chroma database ready at: {persist_dir}")
print(f"Total documents indexed in Chroma: {vectorstore._collection.count()}")

# Setup retriever (top 5 most relevant animes)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

Chroma database ready at: chroma_db
Total documents indexed in Chroma: 269


In [7]:
# Test query to verify complete document retrieval
test_query = "martial arts ninja shounen"
sample_results = retriever.invoke(test_query)

print(f"Retrieved {len(sample_results)} candidates for: '{test_query}'\n")
for i, doc in enumerate(sample_results[:3], 1):
    print(f"[{i}] {doc.metadata.get('title')} (Score: {doc.metadata.get('score')})")
    print(f"    Genres: {doc.metadata.get('genres')}")
    overview_text = doc.page_content.split('Overview: ')[-1].split('\nGenres:')[0]
    print(f"    Synopsis: {overview_text[:120]}...\n")

Retrieved 5 candidates for: 'martial arts ninja shounen'

[1] Ninin ga Shinobuden (Score: 6.92)
    Genres: Comedy, Parody, Martial Arts
    Synopsis: Kaede is a normal school girl who was studying for her exams for school when suddenly she is interrupted by Shinobu, a g...

[2] Rurouni Kenshin: Meiji Kenkaku Romantan - Ishinshishi e no Chinkonka (Score: 7.56)
    Genres: Samurai, Historical, Drama, Shounen
    Synopsis: The war against the Tokugawa Shogunate ended years ago. But there are some who are not happy with the outcome. Shigure T...

[3] Rurouni Kenshin: Meiji Kenkaku Romantan - Tsuioku-hen (Score: 8.73)
    Genres: Action, Historical, Drama, Romance, Martial Arts, Samurai, Shounen
    Synopsis: hen mankind's savagery surpasses his fear of death, there is little hope for those who wish to live honest lives. Beneat...



## 4. LangChain Tool Definition & Agentic Workflow

We define `@tool def anime_retriever_tool(query: str)` which allows the LLM to dynamically search the vector store whenever the user asks for recommendations.

In [8]:
@tool
def anime_retriever_tool(query: str) -> str:
    """
    Searches the anime vector database for relevant anime based on title, genre, synopsis, or themes.
    Returns complete information for matching anime titles.
    """
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant anime found."

    formatted = []
    for d in docs:
        title = d.metadata.get("title", "Unknown")
        score = d.metadata.get("score", "N/A")
        genres = d.metadata.get("genres", "N/A")
        formatted.append(f"Title: {title}\nScore: {score}\nGenres: {genres}\n{d.page_content}")

    return "\n\n---\n\n".join(formatted)

# Bind tool to model
model_with_tools = model.bind_tools([anime_retriever_tool])
print("Tool bound to model successfully.")

Tool bound to model successfully.


In [9]:
from langchain_core.messages import SystemMessage

def run_anime_recommender(user_query: str):
    """
    Executes the 3-step tool-calling workflow:
    1. Model decides whether to call anime_retriever_tool.
    2. Tool executes search on Chroma vector store & prints retrieved anime.
    3. Model receives tool output and generates the final personalized recommendation.
    """
    print(f"User Query: {user_query}")
    print("-" * 50)
    
    # System prompt instructs the model to synthesize final answer after 1 tool call
    messages = [
        SystemMessage(content=(
            "You are an expert anime recommender. When asked for recommendations, "
            "use the anime_retriever_tool to find relevant anime. Once you receive the tool results, "
            "do NOT call any further tools; immediately synthesize a clear, well-formatted recommendation."
        )),
        HumanMessage(content=user_query)
    ]

    # Step 1: Model inspects query and decides tool invocation
    ai_msg = model_with_tools.invoke(messages)
    messages.append(ai_msg)

    # Step 2: Handle tool calls if requested
    if ai_msg.tool_calls:
        for tool_call in ai_msg.tool_calls:
            print(f"-> Calling tool: {tool_call['name']}({tool_call['args']})")
            tool_output = anime_retriever_tool.invoke(tool_call["args"])
            
            # Print what the retriever returned
            print("\n=== Retrieved Anime from Vector Database ===")
            print(tool_output)
            print("-" * 50)
            
            messages.append(
                ToolMessage(
                    content=tool_output,
                    tool_call_id=tool_call["id"]
                )
            )

        # Step 3: Final response from model synthesizing the retrieved anime
        final_response = model_with_tools.invoke(messages)
        
        # Fallback to reasoning_content if content is empty
        recommendation = final_response.content or final_response.additional_kwargs.get("reasoning_content", "")
        
        print("\n=== Final Recommendation ===")
        print(recommendation)
        return final_response
    else:
        print("\n=== Direct Response (No tool called) ===")
        print(ai_msg.content)
        return ai_msg

In [10]:
# Query 1: Recommendation similar to Naruto
response_1 = run_anime_recommender("Can you suggest me some anime like Naruto?")

User Query: Can you suggest me some anime like Naruto?
--------------------------------------------------
-> Calling tool: anime_retriever_tool({'query': 'anime similar to Naruto'})

=== Retrieved Anime from Vector Database ===
Title: Naruto
Score: 7.91
Genres: Action, Adventure, Comedy, Super Power, Martial Arts, Shounen
Title: Naruto
Overview: oments prior to Naruto Uzumaki's birth, a huge demon known as the Kyuubi, the Nine-Tailed Fox, attacked Konohagakure, the Hidden Leaf Village, and wreaked havoc. In order to put an end to the Kyuubi's rampage, the leader of the village, the Fourth Hokage, sacrificed his life and sealed the monstrous beast inside the newborn Naruto. Now, Naruto is a hyperactive and knuckle-headed ninja still living in Konohagakure. Shunned because of the Kyuubi inside him, Naruto struggles to find his place in the village, while his burning desire to become the Hokage of Konohagakure leads him not only to some great new friends, but also some deadly foes.
Genres

In [11]:
# Query 2: Recommendation for space / sci-fi anime
response_2 = run_anime_recommender("I am looking for a thrilling space sci-fi anime with deep storytelling and great scores.")

User Query: I am looking for a thrilling space sci-fi anime with deep storytelling and great scores.
--------------------------------------------------
-> Calling tool: anime_retriever_tool({'query': 'space sci-fi anime deep storytelling great score'})

=== Retrieved Anime from Vector Database ===
Title: Xenosaga The Animation
Score: 6.37
Genres: Action, Mecha, Sci-Fi, Shounen, Space
Title: Xenosaga The Animation
Overview: The year is T.C. 4767. Four thousand years have passed since humanity abandoned its birthplace, the planet Earth. Beset by the hostile alien Gnosis, mankind is now scrambling to find ways to defeat this threat to their existence. The development of KOS-MOS (a specialized android with amazing capabilities) by Vector engineer Shion Uzuki was one response to the threat. But when their ship is destroyed by the Gnosis, Shion and her companions find themselves thrust into the middle of a battle with no clear sides... (Source: Anime-Planet)
Genres: Action, Mecha, Sci-Fi, Sh